In [93]:
import pandas as pd

In [94]:
# Load Datasets

online = pd.read_csv('MultipleDataSources/online_sales.csv')
store = pd.read_csv('MultipleDataSources/store_sales.csv')
email = pd.read_csv('MultipleDataSources/email_campaign.csv')

In [95]:
# Standardize column names
online.columns = online.columns.str.lower().str.strip()
store.columns = store.columns.str.lower().str.strip()
email.columns = email.columns.str.lower().str.strip()

Clean Online Sales

In [96]:
# Remove currency symbols and convert to numeric
online['revenue'] = online['revenue'].replace('[\$]', '', regex = True)
online['revenue'] = pd.to_numeric(online['revenue'], errors = 'coerce')

In [98]:
# standardize date format
online['purchase_date'] = pd.to_datetime(online['purchase_date'], errors = 'coerce')
#online['purchase_date'] = pd.to_numeric(online['purchase_date'], errors = 'coerce')

# errors='coerce' is used during data type conversion to prevent pipeline failures.
# Instead of throwing an error for invalid values, it converts them to NaN, 
# allowing us to identify and handle problematic records systematically.

In [ ]:
# Handle missing Customer Id
online['customer_id'] = online['customer_id'].fillna('Unknown')

In [ ]:
# Remove Duplicates
online = online.drop_duplicates()

Clean Store details

In [ ]:
store['purchase_date'] = pd.to_datetime(store['purchase_date'], errors='coerce')
store['revenue'] = pd.to_numeric(store['revenue'], errors='coerce')

# Remove negative revenue
store = store[store['revenue'] >= 0]

# Remove duplicates
store = store.drop_duplicates()

# Since store has no customer_id, create placeholder
store['customer_id'] = store.get('customer_id', 'Store_Unknown')

Clean Email Campaign Data

In [ ]:
# Standardize email format
email['email'] = email['email'].str.lower().str.strip()

# Convert open_rate to numeric
email['open_rate'] = pd.to_numeric(email['open_rate'], errors='coerce')

# Handle missing open_rate (fill with median)
email['open_rate'] = email['open_rate'].fillna(email['open_rate'].median())

# Convert engagement_level to numeric scale
engagement_map = {
    "low": 1,
    "medium": 2,
    "high": 3
}

email['engagement_level'] = email['engagement_level'].str.lower()
email['engagement_score'] = email['engagement_level'].map(engagement_map)

Merge Sales Data

In [ ]:
sales = pd.concat([online, store], ignore_index=True)

sales = pd.merge(sales, email, on='email', how='left')

# ignore_index=True is used in concatenation to reset row indexing after combining multiple datasets. 
# Without it, duplicate indices from different sources could cause data handling issues. 
# It ensures a clean, sequential index in the final integrated dataset.

'''Online and store datasets were concatenated because they represent the same entity—sales transactions—
collected from different channels. However, the email campaign dataset has a different structure and grain, 
representing customer engagement rather than transactions.
 Therefore, it should be merged using a common key rather than concatenated.
'''

In [ ]:
# Remove rows with invalid dates
sales = sales[sales['purchase_date'].notnull()]

Feature Engineering

In [ ]:
sales['year'] = sales['purchase_date'].dt.year
sales['month'] = sales['purchase_date'].dt.month
sales['day'] = sales['purchase_date'].dt.day

Data Quality Checks

In [ ]:
print("\nData Quality Summary:")
print("Missing values:\n", sales.isnull().sum())
print("Duplicate rows:", sales.duplicated().sum())
print("Negative revenue count:", (sales['revenue'] < 0).sum())

Save Clean Data

In [ ]:
sales.to_csv("MultipleDataSources/sales_cleaned.csv", index=False)

print("\nETL process completed successfully.")